
<center><h1>The Annotated Transformer</h1> </center>


<center>
<p><a href="https://arxiv.org/abs/1706.03762">Attention is All You Need
</a></p>
</center>

<img src="images/aiayn.png" width="70%"/>

* *v2022: Austin Huang, Suraj Subramanian, Jonathan Sum, Khalid Almubarak,
   and Stella Biderman.*
* *[Original](https://nlp.seas.harvard.edu/2018/04/03/attention.html):
   [Sasha Rush](http://rush-nlp.com/).*


The Transformer has been on a lot of
people's minds over the last <s>year</s> five years.
This post presents an annotated version of the paper in the
form of a line-by-line implementation. It reorders and deletes
some sections from the original paper and adds comments
throughout. This document itself is a working notebook, and should
be a completely usable implementation.
Code is available
[here](https://github.com/harvardnlp/annotated-transformer/).


<h3> Table of Contents </h3>
<ul>
<li><a href="#prelims">Prelims</a></li>
<li><a href="#background">Background</a></li>
<li><a href="#part-1-model-architecture">Part 1: Model Architecture</a></li>
<li><a href="#model-architecture">Model Architecture</a><ul>
<li><a href="#encoder-and-decoder-stacks">Encoder and Decoder Stacks</a></li>
<li><a href="#position-wise-feed-forward-networks">Position-wise Feed-Forward
Networks</a></li>
<li><a href="#embeddings-and-softmax">Embeddings and Softmax</a></li>
<li><a href="#positional-encoding">Positional Encoding</a></li>
<li><a href="#full-model">Full Model</a></li>
<li><a href="#inference">Inference:</a></li>
</ul></li>
<li><a href="#part-2-model-training">Part 2: Model Training</a></li>
<li><a href="#training">Training</a><ul>
<li><a href="#batches-and-masking">Batches and Masking</a></li>
<li><a href="#training-loop">Training Loop</a></li>
<li><a href="#training-data-and-batching">Training Data and Batching</a></li>
<li><a href="#hardware-and-schedule">Hardware and Schedule</a></li>
<li><a href="#optimizer">Optimizer</a></li>
<li><a href="#regularization">Regularization</a></li>
</ul></li>
<li><a href="#a-first-example">A First Example</a><ul>
<li><a href="#synthetic-data">Synthetic Data</a></li>
<li><a href="#loss-computation">Loss Computation</a></li>
<li><a href="#greedy-decoding">Greedy Decoding</a></li>
</ul></li>
<li><a href="#part-3-a-real-world-example">Part 3: A Real World Example</a>
<ul>
<li><a href="#data-loading">Data Loading</a></li>
<li><a href="#iterators">Iterators</a></li>
<li><a href="#training-the-system">Training the System</a></li>
</ul></li>
<li><a href="#additional-components-bpe-search-averaging">Additional
Components: BPE, Search, Averaging</a></li>
<li><a href="#results">Results</a><ul>
<li><a href="#attention-visualization">Attention Visualization</a></li>
<li><a href="#encoder-self-attention">Encoder Self Attention</a></li>
<li><a href="#decoder-self-attention">Decoder Self Attention</a></li>
<li><a href="#decoder-src-attention">Decoder Src Attention</a></li>
</ul></li>
<li><a href="#conclusion">Conclusion</a></li>
</ul>

# Prelims

<a href="#background">Skip</a>

In [1]:
# !pip install -r requirements.txt

In [2]:
# # Uncomment for colab
# #
# !pip install -q torchdata==0.3.0 torchtext==0.12 spacy==3.2 altair GPUtil
# !python -m spacy download de_core_news_sm
# !python -m spacy download en_core_web_sm

In [1]:
import os
from os.path import exists
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
import math
import copy
import time
from torch.optim.lr_scheduler import LambdaLR
import pandas as pd
import altair as alt
# from torchtext.data.functional import to_map_style_dataset
from torch.utils.data import DataLoader
# from torchtext.vocab import build_vocab_from_iterator
# import torchtext.datasets as datasets
# import spacy
# import GPUtil
import warnings
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP


# Set to False to skip notebook execution (e.g. for debugging)
warnings.filterwarnings("ignore")
RUN_EXAMPLES = True

In [2]:
# Some convenience helper functions used throughout the notebook


def is_interactive_notebook():
    return __name__ == "__main__"


def show_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        return fn(*args)


def execute_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        fn(*args)


class DummyOptimizer(torch.optim.Optimizer):
    def __init__(self):
        self.param_groups = [{"lr": 0}]
        None

    def step(self):
        None

    def zero_grad(self, set_to_none=False):
        None


class DummyScheduler:
    def step(self):
        None

> My comments are blockquoted. The main text is all from the paper itself.

# Background


The goal of reducing sequential computation also forms the
foundation of the Extended Neural GPU, ByteNet and ConvS2S, all of
which use convolutional neural networks as basic building block,
computing hidden representations in parallel for all input and
output positions. In these models, the number of operations required
to relate signals from two arbitrary input or output positions grows
in the distance between positions, linearly for ConvS2S and
logarithmically for ByteNet. This makes it more difficult to learn
dependencies between distant positions. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of
reduced effective resolution due to averaging attention-weighted
positions, an effect we counteract with Multi-Head Attention.

Self-attention, sometimes called intra-attention is an attention
mechanism relating different positions of a single sequence in order
to compute a representation of the sequence. Self-attention has been
used successfully in a variety of tasks including reading
comprehension, abstractive summarization, textual entailment and
learning task-independent sentence representations. End-to-end
memory networks are based on a recurrent attention mechanism instead
of sequencealigned recurrence and have been shown to perform well on
simple-language question answering and language modeling tasks.

To the best of our knowledge, however, the Transformer is the first
transduction model relying entirely on self-attention to compute
representations of its input and output without using sequence
aligned RNNs or convolution.

# Part 1: Model Architecture

# Model Architecture


Most competitive neural sequence transduction models have an
encoder-decoder structure
[(cite)](https://arxiv.org/abs/1409.0473). Here, the encoder maps an
input sequence of symbol representations $(x_1, ..., x_n)$ to a
sequence of continuous representations $\mathbf{z} = (z_1, ...,
z_n)$. Given $\mathbf{z}$, the decoder then generates an output
sequence $(y_1,...,y_m)$ of symbols one element at a time. At each
step the model is auto-regressive
[(cite)](https://arxiv.org/abs/1308.0850), consuming the previously
generated symbols as additional input when generating the next.

In [3]:
# first, we define the total architecture of EncoderDecoder

class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture. Base for this and many
    other models.
    """

    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def forward(self, src, tgt, src_mask, tgt_mask):
        """
        Take in and process masked src and target sequences.
        here src and tgt are the token id matrix with shape (batch_size, sequence_length)
        (each id num represents a one-hot code)
        src_mask, tgt_mask are the mask matrix with the same shape
        where 1 represents no mask while 0 represents mask.
        For encoder, we need src_mask for padding tokens
        For decoder, since there is a cross-attention mechanism, 
        both src_mask and tgt_mask are needed to compute attention.
        """
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

In [4]:
class Generator(nn.Module):
    "Define standard linear + softmax generation step."
    "This module is to predict the probability of each output token embedding"
    def __init__(self, d_model, vocab):
        super(Generator, self).__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x):
        return log_softmax(self.proj(x), dim=-1)


The Transformer follows this overall architecture using stacked
self-attention and point-wise, fully connected layers for both the
encoder and decoder, shown in the left and right halves of Figure 1,
respectively.

![](images/ModalNet-21.png)

## Encoder and Decoder Stacks

### Encoder

The encoder is composed of a stack of $N=6$ identical layers.

In [5]:
def clones(module, N):
    "Produce N identical layers."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [6]:
# define the total architecture of Encoder

class Encoder(nn.Module):
    "Core encoder is a stack of N layers"

    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        """
        Pass the input (and mask) through each layer in turn.
        for each layer, it will first perform LayerNorm to make sure the scale normalization.
        so we need the final LayerNorm to perform to the final output the N layers.
        """
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


We employ a residual connection
[(cite)](https://arxiv.org/abs/1512.03385) around each of the two
sub-layers, followed by layer normalization
[(cite)](https://arxiv.org/abs/1607.06450).

In [7]:
class LayerNorm(nn.Module):
    "Construct a layernorm module (See citation for details)."

    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        """
        LayerNorm is perform to each token embedding seperately.
        It first compute the embedding's mean and std, then normalize it by {x}=(x-u)/s
        After that, it then use two parameters a and b. a is to scale each dimension
        while b is to shift the center.
        This can be seen intuitively. a new center vector b + (a * {x}) 
        we can understand by this way: 
            1. each embedding is first normalized seperately: {x}=(x-u)/s
            2. all embeddings are shaped as a whole: first scale each dimension and then move the center
        eps is for stable (do not divided by zero!)
        """
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2


That is, the output of each sub-layer is $\mathrm{LayerNorm}(x +
\mathrm{Sublayer}(x))$, where $\mathrm{Sublayer}(x)$ is the function
implemented by the sub-layer itself.  We apply dropout
[(cite)](http://jmlr.org/papers/v15/srivastava14a.html) to the
output of each sub-layer, before it is added to the sub-layer input
and normalized.

To facilitate these residual connections, all sub-layers in the
model, as well as the embedding layers, produce outputs of dimension
$d_{\text{model}}=512$.

In [8]:
# to define the architecture of each layer in Encoder
# we need to use the residual connection
# for each encoder layer, we have two sublayer: attention and MLP
# since each residual connection for a sublayer, we need two residual connections
# to perform sublayer, we first need to perform Laynorm to have unified scale
# after LayerNorm and sublayer, we need dropout and residual connection
# keep in mind that put LayerNorm first before other operation

class SublayerConnection(nn.Module):
    """
    A residual connection followed by a layer norm.
    Note for code simplicity the norm is first as opposed to last.
    """

    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        "Apply residual connection to any sublayer with the same size."
        "sublayer is a operator module (a function)"
        return x + self.dropout(sublayer(self.norm(x)))


Each layer has two sub-layers. The first is a multi-head
self-attention mechanism, and the second is a simple, position-wise
fully connected feed-forward network.

In [9]:
# define the architecture of each layer in Encoder
# self_attn and feed_forward are two function modules

class EncoderLayer(nn.Module):
    "Encoder is made up of self-attn and feed forward (defined below)"

    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        "Follow Figure 1 (left) for connections."
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)

### Decoder

The decoder is also composed of a stack of $N=6$ identical layers.


In [10]:
# define the total architecture of Decoder

class Decoder(nn.Module):
    "Generic N layer decoder with masking."

    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)


In addition to the two sub-layers in each encoder layer, the decoder
inserts a third sub-layer, which performs multi-head attention over
the output of the encoder stack.  Similar to the encoder, we employ
residual connections around each of the sub-layers, followed by
layer normalization.

In [11]:
class DecoderLayer(nn.Module):
    "Decoder is made of self-attn, src-attn, and feed forward (defined below)"

    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        "Follow Figure 1 (right) for connections."
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)


We also modify the self-attention sub-layer in the decoder stack to
prevent positions from attending to subsequent positions.  This
masking, combined with fact that the output embeddings are offset by
one position, ensures that the predictions for position $i$ can
depend only on the known outputs at positions less than $i$.

In [12]:
def subsequent_mask(size):
    "Mask out subsequent positions."
    "triu set the down-side to 0, with diagonal=1, the diagonal elements would also be set to 0"
    attn_shape = (1, size, size)
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(
        torch.uint8
    )
    return subsequent_mask == 0


> Below the attention mask shows the position each tgt word (row) is
> allowed to look at (column). Words are blocked for attending to
> future words during training.

In [13]:
def example_mask():
    LS_data = pd.concat(
        [
            pd.DataFrame(
                {
                    "Subsequent Mask": subsequent_mask(20)[0][x, y].flatten(),
                    "Window": y,
                    "Masking": x,
                }
            )
            for y in range(20)
            for x in range(20)
        ]
    )

    return (
        alt.Chart(LS_data)
        .mark_rect()
        .properties(height=250, width=250)
        .encode(
            alt.X("Window:O"),
            alt.Y("Masking:O"),
            alt.Color("Subsequent Mask:Q", scale=alt.Scale(scheme="viridis")),
        )
        .interactive()
    )


show_example(example_mask)

alt.Chart(...)

### Attention

An attention function can be described as mapping a query and a set
of key-value pairs to an output, where the query, keys, values, and
output are all vectors.  The output is computed as a weighted sum of
the values, where the weight assigned to each value is computed by a
compatibility function of the query with the corresponding key.

We call our particular attention "Scaled Dot-Product Attention".
The input consists of queries and keys of dimension $d_k$, and
values of dimension $d_v$.  We compute the dot products of the query
with all keys, divide each by $\sqrt{d_k}$, and apply a softmax
function to obtain the weights on the values.



![](images/ModalNet-19.png)


In practice, we compute the attention function on a set of queries
simultaneously, packed together into a matrix $Q$.  The keys and
values are also packed together into matrices $K$ and $V$.  We
compute the matrix of outputs as:

$$
   \mathrm{Attention}(Q, K, V) = \mathrm{softmax}(\frac{QK^T}{\sqrt{d_k}})V
$$

In [14]:
# keep in mind that in most cases in NLP, we only need to care three-dimensions tensor
# the last two dimensions are what we really care.
# key.transpose(-2, -1) exchange the last two dimensions
# torch.matmul will perform matrix multiplication batch to batch
# d_k = query.size(-1) is the embedding dimension num
# scores.masked_fill(mask == 0, 1e-9) will set index (i,j) where mask[i,j]=0 to large negative num
# therefore after softmax: p_attn = scores.softmax(dim=-1), p_attn[i,j] will be 0
# this is the masked attention mechanism
# here torch.matmul use broadcast. e.g. key.shape = query.shape = (batch_size, h, seq_len, d_k)
# torch.matmul(query, key.transpose(-2, -1)) will give a output.shape = (batch_size, h, seq_len, seq_len)
# where output[i,j] = torch.matmul(query[i,j], key[i,j].transpose(-2, -1))
# original mask.shape = (batch_size, 1, seq_len), to make use of broadcast
# mask.shape should be (batch_size, 1, 1, seq_len), this can be done using mask.unsqueeze(-2)
# so for (batch_size, h, seq_len, seq_len), batch j's attention score(j, h, seq_len, seq_len) will be masked
# since LayerNorm, suppose that q_i and k_i are independent random variables with mean 0 and variance 1
# \sum q_i k_i then will has mean 0 and variance d_k, this large magnitude will push the softmax into regions where
# it has extremely small gradients. to counteract this effect, we scale the dot products by 1/sqrt(d_k)

def attention(query, key, value, mask=None, dropout=None):
    "Compute 'Scaled Dot Product Attention'"
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn


The two most commonly used attention functions are additive
attention [(cite)](https://arxiv.org/abs/1409.0473), and dot-product
(multiplicative) attention.  Dot-product attention is identical to
our algorithm, except for the scaling factor of
$\frac{1}{\sqrt{d_k}}$. Additive attention computes the
compatibility function using a feed-forward network with a single
hidden layer.  While the two are similar in theoretical complexity,
dot-product attention is much faster and more space-efficient in
practice, since it can be implemented using highly optimized matrix
multiplication code.


While for small values of $d_k$ the two mechanisms perform
similarly, additive attention outperforms dot product attention
without scaling for larger values of $d_k$
[(cite)](https://arxiv.org/abs/1703.03906). We suspect that for
large values of $d_k$, the dot products grow large in magnitude,
pushing the softmax function into regions where it has extremely
small gradients (To illustrate why the dot products get large,
assume that the components of $q$ and $k$ are independent random
variables with mean $0$ and variance $1$.  Then their dot product,
$q \cdot k = \sum_{i=1}^{d_k} q_ik_i$, has mean $0$ and variance
$d_k$.). To counteract this effect, we scale the dot products by
$\frac{1}{\sqrt{d_k}}$.



![](images/ModalNet-20.png)


Multi-head attention allows the model to jointly attend to
information from different representation subspaces at different
positions. With a single attention head, averaging inhibits this.

$$
\mathrm{MultiHead}(Q, K, V) =
    \mathrm{Concat}(\mathrm{head_1}, ..., \mathrm{head_h})W^O \\
    \text{where}~\mathrm{head_i} = \mathrm{Attention}(QW^Q_i, KW^K_i, VW^V_i)
$$

Where the projections are parameter matrices $W^Q_i \in
\mathbb{R}^{d_{\text{model}} \times d_k}$, $W^K_i \in
\mathbb{R}^{d_{\text{model}} \times d_k}$, $W^V_i \in
\mathbb{R}^{d_{\text{model}} \times d_v}$ and $W^O \in
\mathbb{R}^{hd_v \times d_{\text{model}}}$.

In this work we employ $h=8$ parallel attention layers, or
heads. For each of these we use $d_k=d_v=d_{\text{model}}/h=64$. Due
to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full
dimensionality.

In [15]:
# query, key and value are all [batch_size, seq_len, d_model] shape
# zip() packs them together, so one linear transformation corresponds to a query, key and value, respectively
# lin(x) do the transformation, lin(x).view(nbatches, -1, self.h, self.d_k) seperates the last dimension
# d_model to self.h x self.d_k, ().transpose(1, 2) will exchange the 1th and 2th
# so the final dimension is (batch_size, h, seq_len, d_k)
# mask.shape = (batch_size, 1, seq_len), to remember this, think about the subsequent_mask
# whose shape = (1, seq_len, seq_len). they should have the three dimensions.
# actually, mask.shape should be (batch_size, seq_len, seq_len), but in src_mask, 
# src_mask(batch_size, i, :) and src_mask(batch_size, j, :) is the same
# so for simple and easy, just let src_mask.shape = (batch_size, 1, seq_len)
# original mask.shape = (batch_size, 1, seq_len), to make use of broadcast
# mask.shape should be (batch_size, 1, 1, seq_len), this can be done using mask.unsqueeze(-2)
# or equivalently, mask.unsqueeze(1)
# after attention, will return x.shape = (batch_size, h, seq_len, d_k)
# by exchange (1, 2), get x.shape = (batch_size, seq_len, h, d_k)
# by view(nbatches, -1, h*d_k), get x.shape = (batch_size, seq_len, d_model)
# here .contiguous() is to make the memory of tensor continious, so .view() can be used
# del query, key and value to save memory
# finally, perform a whole linear transformation to x: self.linears[-1](x)

class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        "Take in model size and number of heads."
        super(MultiHeadedAttention, self).__init__()
        assert d_model % h == 0
        # We assume d_v always equals d_k
        self.d_k = d_model // h
        self.h = h
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        "Implements Figure 2"
        if mask is not None:
            # Same mask applied to all h heads.
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)

        # 1) Do all the linear projections in batch from d_model => h x d_k
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]

        # 2) Apply attention on all the projected vectors in batch.
        x, self.attn = attention(
            query, key, value, mask=mask, dropout=self.dropout
        )

        # 3) "Concat" using a view and apply a final linear.
        x = (
            x.transpose(1, 2)
            .contiguous()
            .view(nbatches, -1, self.h * self.d_k)
        )
        del query
        del key
        del value
        return self.linears[-1](x)

### Applications of Attention in our Model

The Transformer uses multi-head attention in three different ways:
1) In "encoder-decoder attention" layers, the queries come from the
previous decoder layer, and the memory keys and values come from the
output of the encoder.  This allows every position in the decoder to
attend over all positions in the input sequence.  This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence
models such as [(cite)](https://arxiv.org/abs/1609.08144).


2) The encoder contains self-attention layers.  In a self-attention
layer all of the keys, values and queries come from the same place,
in this case, the output of the previous layer in the encoder.  Each
position in the encoder can attend to all positions in the previous
layer of the encoder.


3) Similarly, self-attention layers in the decoder allow each
position in the decoder to attend to all positions in the decoder up
to and including that position.  We need to prevent leftward
information flow in the decoder to preserve the auto-regressive
property.  We implement this inside of scaled dot-product attention
by masking out (setting to $-\infty$) all values in the input of the
softmax which correspond to illegal connections.

## Position-wise Feed-Forward Networks

In addition to attention sub-layers, each of the layers in our
encoder and decoder contains a fully connected feed-forward network,
which is applied to each position separately and identically.  This
consists of two linear transformations with a ReLU activation in
between.

$$\mathrm{FFN}(x)=\max(0, xW_1 + b_1) W_2 + b_2$$

While the linear transformations are the same across different
positions, they use different parameters from layer to
layer. Another way of describing this is as two convolutions with
kernel size 1.  The dimensionality of input and output is
$d_{\text{model}}=512$, and the inner-layer has dimensionality
$d_{ff}=2048$.

In [16]:
# generally, d_ff is four times of d_model

class PositionwiseFeedForward(nn.Module):
    "Implements FFN equation."

    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(self.w_1(x).relu()))

## Embeddings and Softmax

Similarly to other sequence transduction models, we use learned
embeddings to convert the input tokens and output tokens to vectors
of dimension $d_{\text{model}}$.  We also use the usual learned
linear transformation and softmax function to convert the decoder
output to predicted next-token probabilities.  In our model, we
share the same weight matrix between the two embedding layers and
the pre-softmax linear transformation, similar to
[(cite)](https://arxiv.org/abs/1608.05859). In the embedding layers,
we multiply those weights by $\sqrt{d_{\text{model}}}$.

In [17]:
# here self.lut is a look-up table, it accept an id num and return the corresponding embedding.
# one can see this as a full-connected linear layer that takes one-hot code as input. 
# padding id is an all-zero vector, and do not contribute to gradient. it is a fixed padding vector
# each embedding has a norm norm. for example, l2 norm = 1
# \sum x_i^2 = 1, x_i = 1/sqrt(d_model)
# to keep each dimension has a same scale, each one should multiply sqrt(d_model)
# return self.lut(x) * math.sqrt(self.d_model)
# so the model is unrelational with the num of d_model


class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)
        # return self.lut(x)

## Positional Encoding

Since our model contains no recurrence and no convolution, in order
for the model to make use of the order of the sequence, we must
inject some information about the relative or absolute position of
the tokens in the sequence.  To this end, we add "positional
encodings" to the input embeddings at the bottoms of the encoder and
decoder stacks.  The positional encodings have the same dimension
$d_{\text{model}}$ as the embeddings, so that the two can be summed.
There are many choices of positional encodings, learned and fixed
[(cite)](https://arxiv.org/pdf/1705.03122.pdf).

In this work, we use sine and cosine functions of different frequencies:

$$PE_{(pos,2i)} = \sin(pos / 10000^{2i/d_{\text{model}}})$$

$$PE_{(pos,2i+1)} = \cos(pos / 10000^{2i/d_{\text{model}}})$$

where $pos$ is the position and $i$ is the dimension.  That is, each
dimension of the positional encoding corresponds to a sinusoid.  The
wavelengths form a geometric progression from $2\pi$ to $10000 \cdot
2\pi$.  We chose this function because we hypothesized it would
allow the model to easily learn to attend by relative positions,
since for any fixed offset $k$, $PE_{pos+k}$ can be represented as a
linear function of $PE_{pos}$.

In addition, we apply dropout to the sums of the embeddings and the
positional encodings in both the encoder and decoder stacks.  For
the base model, we use a rate of $P_{drop}=0.1$.



In [18]:
# pos_embed.shape = (512, d_model)
# e^{-iwt} = cos(wt) + i sin(wt)
# we use e^{-iwt} to encode position information
# consider the most simple case, where d_model = 2, pos_embed.shape = (512, 2)
# set t = 0 to t = 511
# pos_embed[t1] = (cos(w t1), sin(w t1))
# pos_embed[t2] = (cos(w t2), sin(w t2))
# if i is close to j, pos_embed[t1] is close to pos_embed[t2]
# e^{-iw(t2-t1)} \to 0
# now we have d_model = 512, so we have to set different frequence w
# w0 = 0.98 = 1 / (10000)^{1/512},  w_j = 0.98^j, so the frequence w will tend to be smaller and smaller
# as such, pos_embed can capture the position relation
# now we consider what is the relation between pos_embed[p] and pos_embed[p+k] for any fixed offset k
# for a fixed w, we have e^{-iw(p+k)} = e^{-iwp} * e^{-iwk}
# therefore, [e^{-iw1 (p+k)}, e^{-iw2 (p+k)}, ..., e^{-iw_256 (p+k)}] 
# = [e^{-iw1 k}, e^{-iw2 k}, ..., e^{-iw_256 k}]  [e^{-iw1 p}, e^{-iw2 p}, ..., e^{-iw_256 p}]
# this means that, for a fix offset k, we have pos_embed[p+k] = linear_k(pos_embed[p])
# this linear transformation linear_k is only determined by offset k, nothing about position p, which is reasonable.
# (we chose this function because we hypothesized it would allow the model to easily learn to attend by relative positions,
# since for any fixed offset k, pos_embed[p+k] can be represented as a linear function linear_k of pos_embed[p])
# this means the geometric shape of offset k is isometric. 
# one simple case is pos_embed.shape = (512, 2), and w is small enough to make each position to form a circle
# then the linear transformation linear_k for pos_embed[p] to be pos_embed[p+k] is a rotation with angle(k). 

# exp(log( 1/( 10000^{i/d}) )) = exp( - i/d * log(10000) )

# pos_embed do not need to compute gradient to update
# pos_embed.shape = (1, 512, d_model), so to keep same shape of x.shape = (batch_size, seq_len, d_model)

class PositionalEncoding(nn.Module):
    "Implement the PE function."

    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)


> Below the positional encoding will add in a sine wave based on
> position. The frequency and offset of the wave is different for
> each dimension.

In [19]:
def example_positional():
    pe = PositionalEncoding(20, 0)
    y = pe.forward(torch.zeros(1, 100, 20))

    data = pd.concat(
        [
            pd.DataFrame(
                {
                    "embedding": y[0, :, dim],
                    "dimension": dim,
                    "position": list(range(100)),
                }
            )
            for dim in [4, 5, 6, 7]
        ]
    )

    return (
        alt.Chart(data)
        .mark_line()
        .properties(width=800)
        .encode(x="position", y="embedding", color="dimension:N")
        .interactive()
    )


show_example(example_positional)

alt.Chart(...)


We also experimented with using learned positional embeddings
[(cite)](https://arxiv.org/pdf/1705.03122.pdf) instead, and found
that the two versions produced nearly identical results.  We chose
the sinusoidal version because it may allow the model to extrapolate
to sequence lengths longer than the ones encountered during
training.

## Full Model

> Here we define a function from hyperparameters to a full model.

In [20]:
# define model
# embedding dimension is 512
# attention head is 8
# forward middle layer is 512*4 = 2048
# dropout rate is all set to 0.1
# for encoder and decoder, there are 6 layers
# src_vocab and tgt_vocab are the vocabulary num of source and target language
# everything about model have been defined now

# keep in mind that in python, everything is class and therefore object

# to make the model structure 模块化, we define the key module and use copy.deepcopy to reuse the define module
# to define module class, those who have their own parameters should be defined as class
# such as MultiHeadedAttention (h, linear x 4, d_k), PositionwiseFeedForward(linear x 2, dropout)
# Embedding(embedding look up table), PositionEncoding(position table)
# besides, this module class should realize a forward function to use their defined parameter to compute and encode
# when call this class, it will use this forward function to compute with the input parameter
# For those compute mechanism, such as attention when given query, key and value, this case should be 
# defined as a function


# key module class: (from down to up)
# 1. Embeddings(d_model, src_vocab)
# 2. Embeddings(d_model, tgt_vocab)
# 3. position = PositionalEncoding(d_model, dropout)
# 4. attn = MultiHeadedAttention(h, d_model)
# 5. ff = PositionwiseFeedForward(d_model, d_ff, dropout)
# 6. LayerNorm(size)
# 7. EncoderLayer(d_model, attn, ff, dropout) [use sublayer]
# 8. DecoderLayer(d_model, attn, attn, ff, dropout) [use sublayer]
# 9. Encoder(EncoderLayer, N)
#10. Decoder(DecoderLayer, N)
#11. Generator(d_model, tgt_vocab)
#12. EncoderDecoder(Encoder, Decoder, nn.Sequential(Embeddings_src, position), nn.Sequential(Embeddings_tgt, position), Generator)

# here, attn, ff, position can be reused
# since we use nn.module, it will have a method model.parameters()
# we use nn.init.xavier_uniform_ to initialize 
# The Xavier initialization aims to maintain the variance of the weights and gradients consistent across layers to 
# prevent issues like vanishing or exploding gradients during training.
# U(-a, a): a = sqrt( 6/(units_in + units_out) )
# variance = a^2/3 = 2/(units_in + units_out) = ( (units_in + units_out)/2 )^{-1}
# more units, less variance (1/units_num), keep vector norm same
# mean(x_i) = 0, mean(x_i^2) = 1/units_num
# ||x||^2 = sum(x_i^2), mean(||x||^2) = sum(mean(x_i^2)) = units_num * (1/units_num) = 1
# for units_in = units_out, the mean of ||x||^2 is 1.

def make_model(
    src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1
):
    "Helper: Construct a model from hyperparameters."
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab),
    )

    # This was important from their code.
    # Initialize parameters with Glorot / fan_avg.
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model

## Inference:

> Here we make a forward step to generate a prediction of the
model. We try to use our transformer to memorize the input. As you
will see the output is randomly generated due to the fact that the
model is not trained yet. In the next tutorial we will build the
training function and try to train our model to memorize the numbers
from 1 to 10.

In [25]:
# be careful! src.to(device) is not move src to device, but return a new src.
# test for gpu tranning

# test_model = make_model(10, 10, 2)
# device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# device
# test_model.to(device)
# src = torch.tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]], dtype=torch.long, device=device)
# src_mask = torch.ones(1, 1, 10, device = device)
# test_model.encode(src, src_mask)

In [46]:
# input src is two dim tensor: shape = (batch_size, seq_len)
# input src_embed is three dim tensor: shape = (batch_size, seq_len, embed_dim)
# the attention score is three dim tensor: shape = (batch_size, seq_len, seq_len)
# src_mask should keep the same dim: shape = (batch_size, 1, seq_len)
# 使用贪心算法进行推理
# out[:, -1]: every batch's last output embedding
# max, max_id = torch.max
# src_vocab = 10, tgt_vocab = 10, layer_num = 2

def inference_test():
    test_model = make_model(10, 10, 2)

    # move model to gpu
    # device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    device = torch.device("cuda")
    
    test_model.to(device)

    test_model.eval()
    # src = torch.LongTensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]])
    src = torch.tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]], dtype=torch.long, device=device)
    src_mask = torch.ones(1, 1, 10, device=device)

    memory = test_model.encode(src, src_mask)
    ys = torch.zeros(1, 1).type_as(src)

    for i in range(5):
        out = test_model.decode(
            memory, src_mask, ys, subsequent_mask(ys.size(1)).type_as(src.data)
        )
        prob = test_model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat(
            [ys, torch.empty(1, 1).type_as(src.data).fill_(next_word)], dim=1
        )

    print("Example Untrained Model Prediction:", ys)


def run_tests():
    for _ in range(10):
        inference_test()


show_example(run_tests)

Example Untrained Model Prediction: tensor([[0, 4, 9, 9, 9, 9]], device='cuda:0')
Example Untrained Model Prediction: tensor([[0, 8, 8, 8, 8, 8]], device='cuda:0')
Example Untrained Model Prediction: tensor([[0, 2, 2, 2, 2, 2]], device='cuda:0')
Example Untrained Model Prediction: tensor([[0, 1, 0, 4, 1, 0]], device='cuda:0')
Example Untrained Model Prediction: tensor([[0, 2, 3, 8, 2, 3]], device='cuda:0')
Example Untrained Model Prediction: tensor([[0, 7, 7, 7, 7, 7]], device='cuda:0')
Example Untrained Model Prediction: tensor([[0, 7, 2, 3, 3, 3]], device='cuda:0')
Example Untrained Model Prediction: tensor([[0, 1, 1, 1, 1, 1]], device='cuda:0')
Example Untrained Model Prediction: tensor([[0, 2, 2, 2, 2, 2]], device='cuda:0')
Example Untrained Model Prediction: tensor([[0, 3, 4, 8, 3, 4]], device='cuda:0')


# Part 2: Model Training

# Training

This section describes the training regime for our models.


> We stop for a quick interlude to introduce some of the tools
> needed to train a standard encoder decoder model. First we define a
> batch object that holds the src and target sentences for training,
> as well as constructing the masks.

## Batches and Masking

In [49]:
# src, tgt: shape = (batch_size, seq_len)
# self.src_mask = (src != pad).unsqueeze(-2): set pad_id token's mask to 0
# after .unsqueeze(-2): mask.shape = (batch_size, 1, seq_len)
# ntokens: generate token num
# tgt_mask: make_std_mask(tgt, pad): combine subsequent_mask (shape=(1, seq_len, seq_len)) 
# and padding mask (shape=(batch_size, 1, seq_len)) together

class Batch:
    """Object for holding a batch of data with mask during training."""

    def __init__(self, src, tgt=None, pad=2):  # 2 = <blank>
        self.src = src
        self.src_mask = (src != pad).unsqueeze(-2)
        if tgt is not None:
            self.tgt = tgt[:, :-1]
            self.tgt_y = tgt[:, 1:]
            self.tgt_mask = self.make_std_mask(self.tgt, pad)
            self.ntokens = (self.tgt_y != pad).data.sum()

    @staticmethod
    def make_std_mask(tgt, pad):
        "Create a mask to hide padding and future words."
        tgt_mask = (tgt != pad).unsqueeze(-2)
        tgt_mask = tgt_mask & subsequent_mask(tgt.size(-1)).type_as(
            tgt_mask.data
        )
        return tgt_mask


> Next we create a generic training and scoring function to keep
> track of loss. We pass in a generic loss compute function that
> also handles parameter updates.

## Training Loop

In [50]:
class TrainState:
    """Track number of steps, examples, and tokens processed"""

    step: int = 0  # Steps in the current epoch
    accum_step: int = 0  # Number of gradient accumulation steps
    samples: int = 0  # total # of examples used
    tokens: int = 0  # total # of tokens processed

In [51]:
def run_epoch(
    data_iter,
    model,
    loss_compute,
    optimizer,
    scheduler,
    mode="train",
    accum_iter=1,
    train_state=TrainState(),
):
    """Train a single epoch"""
    start = time.time()
    total_tokens = 0
    total_loss = 0
    tokens = 0
    n_accum = 0
    for i, batch in enumerate(data_iter):
        out = model.forward(
            batch.src, batch.tgt, batch.src_mask, batch.tgt_mask
        )
        loss, loss_node = loss_compute(out, batch.tgt_y, batch.ntokens)
        # loss_node = loss_node / accum_iter
        if mode == "train" or mode == "train+log":
            loss_node.backward()
            train_state.step += 1
            train_state.samples += batch.src.shape[0]
            train_state.tokens += batch.ntokens
            if i % accum_iter == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                n_accum += 1
                train_state.accum_step += 1
            scheduler.step()

        total_loss += loss
        total_tokens += batch.ntokens
        tokens += batch.ntokens
        if i % 40 == 1 and (mode == "train" or mode == "train+log"):
            lr = optimizer.param_groups[0]["lr"]
            elapsed = time.time() - start
            print(
                (
                    "Epoch Step: %6d | Accumulation Step: %3d | Loss: %6.2f "
                    + "| Tokens / Sec: %7.1f | Learning Rate: %6.1e"
                )
                % (i, n_accum, loss / batch.ntokens, tokens / elapsed, lr)
            )
            start = time.time()
            tokens = 0
        del loss
        del loss_node
    return total_loss / total_tokens, train_state

## Training Data and Batching

We trained on the standard WMT 2014 English-German dataset
consisting of about 4.5 million sentence pairs.  Sentences were
encoded using byte-pair encoding, which has a shared source-target
vocabulary of about 37000 tokens. For English-French, we used the
significantly larger WMT 2014 English-French dataset consisting of
36M sentences and split tokens into a 32000 word-piece vocabulary.


Sentence pairs were batched together by approximate sequence length.
Each training batch contained a set of sentence pairs containing
approximately 25000 source tokens and 25000 target tokens.

## Hardware and Schedule

We trained our models on one machine with 8 NVIDIA P100 GPUs.  For
our base models using the hyperparameters described throughout the
paper, each training step took about 0.4 seconds.  We trained the
base models for a total of 100,000 steps or 12 hours. For our big
models, step time was 1.0 seconds.  The big models were trained for
300,000 steps (3.5 days).

## Optimizer

We used the Adam optimizer [(cite)](https://arxiv.org/abs/1412.6980)
with $\beta_1=0.9$, $\beta_2=0.98$ and $\epsilon=10^{-9}$.  We
varied the learning rate over the course of training, according to
the formula:

$$
lrate = d_{\text{model}}^{-0.5} \cdot
  \min({step\_num}^{-0.5},
    {step\_num} \cdot {warmup\_steps}^{-1.5})
$$

This corresponds to increasing the learning rate linearly for the
first $warmup\_steps$ training steps, and decreasing it thereafter
proportionally to the inverse square root of the step number.  We
used $warmup\_steps=4000$.


> Note: This part is very important. Need to train with this setup
> of the model.


> Example of the curves of this model for different model sizes and
> for optimization hyperparameters.

In [52]:
def rate(step, model_size, factor, warmup):
    """
    we have to default the step to 1 for LambdaLR function
    to avoid zero raising to negative power.
    """
    if step == 0:
        step = 1
    return factor * (
        model_size ** (-0.5) * min(step ** (-0.5), step * warmup ** (-1.5))
    )

## Regularization

### Label Smoothing

During training, we employed label smoothing of value
$\epsilon_{ls}=0.1$ [(cite)](https://arxiv.org/abs/1512.00567).
This hurts perplexity, as the model learns to be more unsure, but
improves accuracy and BLEU score.


> We implement label smoothing using the KL div loss. Instead of
> using a one-hot target distribution, we create a distribution that
> has `confidence` of the correct word and the rest of the
> `smoothing` mass distributed throughout the vocabulary.

In [53]:
# define LabelSmoothing
# 1. modify one-hot ground truth, smoothing label, and then return loss
# 2. modify 
# first define the parameter of LabelSmoothing: smoothing, padding_idx, size
# target = (batch_size*seq_len)
# x.shape = (batch_size*seq_len, class_num)

# smoothing/(class_num-2): one for truth class, one for padding class (start token)
# after .unsqueeze(1): target = (batch_size*seq_len, 1)
# x.scatter_(dim=1, target, confident), scatter_() can be used to create one-hot
# zeros(3, 5).scatter_(dim=1, target.unsqueeze(1), 1) 
# nn.KLDivLoss() is a align fun: - <log p_predict, p_true>
# padding_idx is used for a special token (e.g. start symbol)
# true_dist[:, self.padding_idx] should be no probability
# mask is set to ignore the padding_idx target

class LabelSmoothing(nn.Module):
    "Implement label smoothing."

    def __init__(self, size, padding_idx, smoothing=0.0):
        super(LabelSmoothing, self).__init__()
        self.criterion = nn.KLDivLoss(reduction="sum")
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.size = size
        self.true_dist = None

    def forward(self, x, target):
        assert x.size(1) == self.size
        true_dist = x.data.clone()
        true_dist.fill_(self.smoothing / (self.size - 2))
        true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0
        mask = torch.nonzero(target.data == self.padding_idx)
        if mask.dim() > 0:
            true_dist.index_fill_(0, mask.squeeze(), 0.0)
        self.true_dist = true_dist
        return self.criterion(x, true_dist.clone().detach())


> Here we can see an example of how the mass is distributed to the
> words based on confidence.


> Label smoothing actually starts to penalize the model if it gets
> very confident about a given choice.

# A First  Example

> We can begin by trying out a simple copy-task. Given a random set
> of input symbols from a small vocabulary, the goal is to generate
> back those same symbols.

## Synthetic Data

In [63]:
# generate data, given batch_size and nbatches
# data[:, 0] = 1: begin token, class token
# 

def data_gen(V, batch_size, nbatches):
    "Generate random data for a src-tgt copy task."
    device = torch.device("cuda")
    for i in range(nbatches):
        data = torch.randint(1, V, size=(batch_size, 10), device=device)
        data[:, 0] = 1
        src = data.requires_grad_(False).clone().detach()
        tgt = data.requires_grad_(False).clone().detach()
        yield Batch(src, tgt, -1)

## Loss Computation

In [64]:
# x.shape = (batch_size, seq_len, class_num)
# y.shape = (batch_size, seq_len)
# after .view(), x.contiguous().view(-1, x.size(-1)): shape = (batch_size * seq_len, class_num)
# y.contiguous().view(-1): shape = (batch_size * seq_len)
# self.criterion(x, y)

class SimpleLossCompute:
    "A simple loss compute and train function."

    def __init__(self, generator, criterion):
        self.generator = generator
        self.criterion = criterion

    def __call__(self, x, y, norm):
        x = self.generator(x)
        sloss = (
            self.criterion(
                x.contiguous().view(-1, x.size(-1)), y.contiguous().view(-1)
            )
            / norm
        )
        return sloss.data * norm, sloss

## Greedy Decoding

> This code predicts a translation using greedy decoding for simplicity.

In [66]:
def greedy_decode(model, src, src_mask, max_len, start_symbol):
    memory = model.encode(src, src_mask)
    ys = torch.zeros(1, 1).fill_(start_symbol).type_as(src.data)
    for i in range(max_len - 1):
        out = model.decode(
            memory, src_mask, ys, subsequent_mask(ys.size(1)).type_as(src.data)
        )
        prob = model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat(
            [ys, torch.zeros(1, 1).type_as(src.data).fill_(next_word)], dim=1
        )
    return ys

In [92]:
# torch.randint(1, 9, size=(10, 10), device=device)[9:10]

# s = [torch.randint(1, 9, size=(10, 10), device=device)[9:10],
# torch.randint(1, 9, size=(10, 10), device=device)[8:9]]
# torch.cat(s, dim=0)

tensor([[5, 5, 3, 1, 2, 6, 3, 7, 8, 1]], device='cuda:0')

In [ ]:
#     src = torch.tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]], device=device)
#     max_len = src.shape[1]
#     src_mask = torch.ones(1, 1, max_len, device=device)
#     print(src, greedy_decode(model, src, src_mask, max_len=max_len, start_symbol=0))

#     src = torch.tensor([[0, 9, 8, 7, 6, 5, 4, 3, 2, 1]], device=device)
#     max_len = src.shape[1]
#     src_mask = torch.ones(1, 1, max_len, device=device)
#     print(src, greedy_decode(model, src, src_mask, max_len=max_len, start_symbol=0))

#     src = torch.tensor([[0, 5, 4, 1, 2, 8, 9, 3, 6, 7]], device=device)
#     max_len = src.shape[1]
#     src_mask = torch.ones(1, 1, max_len, device=device)
#     print(greedy_decode(model, src, src_mask, max_len=max_len, start_symbol=0))

In [125]:
# Train the simple copy task.

def example_simple_model():
    V = 11
    criterion = LabelSmoothing(size=V, padding_idx=0, smoothing=0.1)
    device = torch.device("cuda")
    model = make_model(V, V, N=2)
    model.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=0.5, betas=(0.9, 0.98), eps=1e-9
    )
    lr_scheduler = LambdaLR(
        optimizer=optimizer,
        lr_lambda=lambda step: rate(
            step, model_size=model.src_embed[0].d_model, factor=1.0, warmup=400
        ),
    )

    # dataset size
    batch_size = 50
    nbatches = 20
    for epoch in range(100):
        model.train()
        run_epoch(
            data_gen(V, batch_size, nbatches),
            model,
            SimpleLossCompute(model.generator, criterion),
            optimizer,
            lr_scheduler,
            mode="train",
        )
        model.eval()
        run_epoch(
            data_gen(V, batch_size, 5),
            model,
            SimpleLossCompute(model.generator, criterion),
            DummyOptimizer(),
            DummyScheduler(),
            mode="eval",
        )[0]

        
    model.eval()
    ntest = 50
    srcs = torch.randint(1, 9, size=(ntest, 10), device=device)
    srcs[:, 0] = 0
    max_len = srcs.shape[1]
    src_mask = torch.ones(1, 1, max_len, device=device)
    tgts = []
    for i in range(ntest):
        src = srcs[i:i+1]
        tgts.append(greedy_decode(model, src, src_mask, max_len=max_len, start_symbol=0))
    print(srcs)
    print(torch.cat(tgts, dim=0))
    print(srcs == torch.cat(tgts, dim=0))
    print((srcs == torch.cat(tgts, dim=0)).sum(dim=1)/10)

In [126]:
execute_example(example_simple_model)

Epoch Step:      1 | Accumulation Step:   2 | Loss:   2.63 | Tokens / Sec: 12562.3 | Learning Rate: 5.5e-06
Epoch Step:      1 | Accumulation Step:   2 | Loss:   1.63 | Tokens / Sec: 10663.0 | Learning Rate: 6.1e-05
Epoch Step:      1 | Accumulation Step:   2 | Loss:   1.41 | Tokens / Sec: 13061.3 | Learning Rate: 1.2e-04
Epoch Step:      1 | Accumulation Step:   2 | Loss:   1.21 | Tokens / Sec: 10775.8 | Learning Rate: 1.7e-04
Epoch Step:      1 | Accumulation Step:   2 | Loss:   0.97 | Tokens / Sec: 12991.8 | Learning Rate: 2.3e-04
Epoch Step:      1 | Accumulation Step:   2 | Loss:   0.68 | Tokens / Sec: 10214.1 | Learning Rate: 2.8e-04
Epoch Step:      1 | Accumulation Step:   2 | Loss:   0.54 | Tokens / Sec: 11735.2 | Learning Rate: 3.4e-04
Epoch Step:      1 | Accumulation Step:   2 | Loss:   0.32 | Tokens / Sec: 16490.6 | Learning Rate: 3.9e-04
Epoch Step:      1 | Accumulation Step:   2 | Loss:   0.21 | Tokens / Sec: 16525.8 | Learning Rate: 4.5e-04
Epoch Step:      1 | Accumul